# 05: EVALUATION (one shot: EyePACS test and APTOS)

**This is the notebook that opens the held-out data.** Everything it does is fixed by
`04_results/locked_config.json`; nothing is selected or tuned here. It refuses to run if
evaluation results already exist, so it runs once.

Reads: the lock; 33 EyePACS test and 33 APTOS prediction files. Records their SHA-256
and the lock's hash alongside the results.

**Primary (tier 1, EyePACS test, 25% burden, rank OR-gate):** number of the 55 pairs whose
seed-averaged advantage over their better constituent is positive, and the mean advantage
across all 55, with a patient-clustered bootstrap (2,000 resamples). Claim supported if the
95% interval of the mean advantage excludes zero. Also: pairs beating the best single
architecture in the roster.

**Also reported:** the full 10-40% curve with bootstrap bands (EyePACS and APTOS); tier 2
witness on test, and its validation-to-test shrinkage (winner's curse, measured); tier 3
engineering pair plus its post-lock control (DenseNet121 binary + plain RETFound binary) and
the paired difference; operating characteristics at 25%; rho (validation) against test
advantage and concordant severe misses, with model-level resampling; raw-probability OR-gate
sensitivity; the weight-invariance grid.

APTOS: curve only for tier 1, image-level bootstrap (no patient ids).

CPU runtime is fine; about 10-15 minutes, mostly the bootstraps. Run cells 1-8.

In [1]:
# Cell 1: auth; load the lock; refuse to run twice
import io, os, json, hashlib, datetime, itertools, time
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')

F_PREDICTIONS = '1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'
F_RESULTS     = '1-HT4ONiwIfuBtC19s7QkDd8Euklh0Xon'
F_FIGURES     = '18eyHRWuqFcUrYiL8nPtpIalV7CreLkMK'

def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid)); done = False
        while not done: _, done = d.next_chunk()
def find_id(name, folder):
    r = svc.files().list(q=f"'{folder}' in parents and name='{name}' and trashed=false",
                         fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None
def up(path, folder, mime):
    svc.files().create(body={'name': os.path.basename(path), 'parents': [folder]},
                       media_body=MediaFileUpload(path, mimetype=mime), fields='id').execute()
    print('  uploaded', os.path.basename(path))
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

if find_id('eval_results_v2.json', F_RESULTS):
    raise SystemExit('Evaluation results already exist. This evaluation is one-shot; do not re-run.')
lid = find_id('locked_config.json', F_RESULTS)
if not lid: raise SystemExit('No locked_config.json found. Lock first.')
dl(lid, 'locked_config.json'); LOCK = json.load(open('locked_config.json')); LOCK_SHA = sha256('locked_config.json')

W = LOCK['cost_function']['weights']
BURDENS = LOCK['operating_point']['curve_burdens']; PRIMARY = LOCK['operating_point']['primary_burden']
T25 = BURDENS.index(PRIMARY)
ARCHS = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
         'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
         'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS = LOCK['seeds']['replicates']
PAIRS = list(itertools.combinations(ARCHS, 2))
WIT = (LOCK['tier2_witness']['model_a'], LOCK['tier2_witness']['model_b'])
ENG = (LOCK['tier3_engineering']['model_a'], LOCK['tier3_engineering']['model_b'])
CTRL = ('densenet121_binary', 'retfound_binary')     # post-lock, pre-test control (see protocol)
RHO_VAL = {tuple(k.split('|')): v for k, v in LOCK['rho']['validation'].items()}
def tag(k): return '' if k == 0 else f'_seed{k}'
print('lock sha256:', LOCK_SHA[:16], '| primary burden', PRIMARY, '| witness', WIT, '| engineering', ENG, '| control', CTRL)

lock sha256: 0338c4fff7f3400c | primary burden 0.25 | witness ('vit_base_5class', 'retfound_adversarial') | engineering ('densenet121_binary', 'retfound_adversarial') | control ('densenet121_binary', 'retfound_binary')


In [2]:
# Cell 2: load held-out predictions (EyePACS test, APTOS)
def load(prefix, split_name):
    Pd, hashes, ref = {}, {}, None
    for a in ARCHS:
        for k in SEEDS:
            fn = f'{prefix}{a}{tag(k)}_v2.csv'
            fid = find_id(fn, F_PREDICTIONS)
            if not fid: raise FileNotFoundError(fn)
            dl(fid, fn); d = pd.read_csv(fn); hashes[fn] = sha256(fn)
            assert set(d['split']) == {split_name}, f'{fn}: unexpected split'
            if ref is None: ref = d[['image_id','patient_id','true_grade','true_label']].reset_index(drop=True)
            assert (d['image_id'].astype(str).values == ref['image_id'].astype(str).values).all(), f'{fn}: row order'
            Pd[(a, k)] = d['predicted_probability'].to_numpy(np.float64)
    return Pd, ref, hashes

PT, REF_T, H_T = load('eyepacs_predictions_test_', 'test')
PA, REF_A, H_A = load('aptos_predictions_external_', 'external')
LAB_T, GRD_T = REF_T['true_label'].to_numpy(int), REF_T['true_grade'].to_numpy(int)
LAB_A, GRD_A = REF_A['true_label'].to_numpy(int), REF_A['true_grade'].to_numpy(int)
for L, G in ((LAB_T, GRD_T), (LAB_A, GRD_A)): assert ((L == 1) == (G >= 2)).all()
print(f'EyePACS test: {len(LAB_T)} images, {REF_T.patient_id.nunique()} patients, referable {LAB_T.mean():.3f}, severe {(GRD_T>=3).sum()}')
print(f'APTOS: {len(LAB_A)} images, referable {LAB_A.mean():.3f}, severe {(GRD_A>=3).sum()}')

EyePACS test: 2900 images, 1450 patients, referable 0.194, severe 130
APTOS: 3662 images, referable 0.406, severe 488


In [3]:
# Cell 3: the locked computation, applied to any index subset (used for point estimates and bootstraps)
def run(idx, PD, LAB, GRD, w=W, mode='rank'):
    lab, grd = LAB[idx], GRD[idx]; n = len(idx)
    missw = np.where(lab == 1, np.where(grd == 4, w['w4'], np.where(grd == 3, w['w3'], w['w2'])), 0).astype(float)
    neg = (lab == 0).astype(float)
    Rs = np.array([int(round(r * n)) for r in BURDENS]) - 1
    tot = missw.sum()
    def curve(score):
        o = np.argsort(-score, kind='stable')
        return tot - np.cumsum(missw[o])[Rs] + w['wfp'] * np.cumsum(neg[o])[Rs]
    sub = {key: p[idx] for key, p in PD.items()}
    sc = {key: rankdata(v) / n for key, v in sub.items()} if mode == 'rank' else sub
    single = {key: curve(v) for key, v in sub.items()}
    arch_mean = np.vstack([np.mean([single[(a, k)] for k in SEEDS], axis=0) for a in ARCHS])
    roster = arch_mean.min(0)
    AC, AR = {}, {}
    for a, b in PAIRS:
        ac, ar = [], []
        for i in SEEDS:
            for j in SEEDS:
                c_or = curve(np.maximum(sc[(a, i)], sc[(b, j)]))
                ac.append(np.minimum(single[(a, i)], single[(b, j)]) - c_or); ar.append(roster - c_or)
        AC[(a, b)], AR[(a, b)] = np.array(ac), np.array(ar)       # 9 x burdens, index = i*3 + j
    return AC, AR, roster, arch_mean

DIAG = [0, 4, 8]    # matched seed combinations (k with k)
def summarize(AC, AR):
    pc = np.vstack([AC[p].mean(0) for p in PAIRS]); pr = np.vstack([AR[p].mean(0) for p in PAIRS])
    return {'pairs_winning': (pc > 0).sum(0), 'mean_adv': pc.mean(0),
            'combos_winning': np.sum([(AC[p] > 0).sum(0) for p in PAIRS], axis=0),
            'pairs_beating_roster': (pr > 0).sum(0),
            'witness_adv_c': AC[WIT].mean(0), 'witness_adv_r': AR[WIT].mean(0),
            'eng_adv': AC[ENG][DIAG].mean(0), 'ctrl_adv': AC[CTRL][DIAG].mean(0),
            'eng_minus_ctrl': AC[ENG][DIAG].mean(0) - AC[CTRL][DIAG].mean(0)}

def ci(samples):
    return np.percentile(samples, 2.5, axis=0), np.percentile(samples, 97.5, axis=0)
print('ready')

ready


In [4]:
# Cell 4: EyePACS test point estimates + patient-clustered bootstrap (2,000)
ALL_T = np.arange(len(LAB_T))
AC_T, AR_T, ROSTER_T, ARCHMEAN_T = run(ALL_T, PT, LAB_T, GRD_T)
PT_T = summarize(AC_T, AR_T)

pid = REF_T['patient_id'].astype(str).to_numpy()
pats = pd.unique(pid)
pat_imgs = [np.where(pid == p)[0] for p in pats]
rng = np.random.default_rng(20260922)
B = 2000; boot_T = {k: [] for k in PT_T}
t0 = time.time()
for b in range(B):
    idx = np.concatenate([pat_imgs[i] for i in rng.integers(0, len(pats), len(pats))])
    s = summarize(*run(idx, PT, LAB_T, GRD_T)[:2])
    for k in s: boot_T[k].append(s[k])
    if (b + 1) % 250 == 0: print(f'  bootstrap {b+1}/{B} ({time.time()-t0:.0f}s)')
boot_T = {k: np.array(v) for k, v in boot_T.items()}

lo, hi = ci(boot_T['mean_adv'])
print(f"\nPRIMARY (EyePACS test, 25%, rank OR): {int(PT_T['pairs_winning'][T25])}/55 pairs win; "
      f"mean advantage {PT_T['mean_adv'][T25]:.1f} (95% CI {lo[T25]:.1f} to {hi[T25]:.1f}); "
      f"claim {'SUPPORTED' if lo[T25] > 0 else 'NOT supported'}")

  bootstrap 250/2000 (52s)
  bootstrap 500/2000 (104s)
  bootstrap 750/2000 (155s)
  bootstrap 1000/2000 (206s)
  bootstrap 1250/2000 (257s)
  bootstrap 1500/2000 (309s)
  bootstrap 1750/2000 (360s)
  bootstrap 2000/2000 (411s)

PRIMARY (EyePACS test, 25%, rank OR): 52/55 pairs win; mean advantage 472.0 (95% CI -72.2 to 702.0); claim NOT supported


In [5]:
# Cell 5: APTOS point estimates + image-level bootstrap (2,000)
ALL_A = np.arange(len(LAB_A))
AC_A, AR_A, ROSTER_A, ARCHMEAN_A = run(ALL_A, PA, LAB_A, GRD_A)
PT_A = summarize(AC_A, AR_A)
rng = np.random.default_rng(20260923)
boot_A = {k: [] for k in PT_A}; t0 = time.time()
for b in range(B):
    s = summarize(*run(rng.integers(0, len(LAB_A), len(LAB_A)), PA, LAB_A, GRD_A)[:2])
    for k in s: boot_A[k].append(s[k])
    if (b + 1) % 250 == 0: print(f'  bootstrap {b+1}/{B} ({time.time()-t0:.0f}s)')
boot_A = {k: np.array(v) for k, v in boot_A.items()}
print('APTOS done')

  bootstrap 250/2000 (65s)
  bootstrap 500/2000 (130s)
  bootstrap 750/2000 (195s)
  bootstrap 1000/2000 (260s)
  bootstrap 1250/2000 (324s)
  bootstrap 1500/2000 (388s)
  bootstrap 1750/2000 (453s)
  bootstrap 2000/2000 (522s)
APTOS done


In [6]:
# Cell 6: secondary analyses on EyePACS test (point estimates unless stated)
# 6a. operating characteristics at 25% (averaged over seed combos)
n = len(LAB_T); R25 = int(round(PRIMARY * n))
RK = {key: rankdata(p) / n for key, p in PT.items()}
def opchar(score):
    pred = np.zeros(n, int); pred[np.argsort(-score, kind='stable')[:R25]] = 1
    tp = ((pred == 1) & (LAB_T == 1)).sum(); fp = ((pred == 1) & (LAB_T == 0)).sum()
    fn = ((pred == 0) & (LAB_T == 1)).sum(); tn = ((pred == 0) & (LAB_T == 0)).sum()
    return {'sensitivity': tp/(tp+fn), 'specificity': tn/(tn+fp), 'ppv': tp/(tp+fp), 'npv': tn/(tn+fn),
            'referral_rate': pred.mean(), 'severe_misses': int(((pred == 0) & (GRD_T >= 3)).sum()),
            'grade4_misses': int(((pred == 0) & (GRD_T == 4)).sum())}
def avg(ds): return {k: round(float(np.mean([d[k] for d in ds])), 4) for k in ds[0]}
best_arch = ARCHS[int(np.argmin(ARCHMEAN_T[:, T25]))]
OPCHAR = {
  f'best_single ({best_arch})': avg([opchar(PT[(best_arch, k)]) for k in SEEDS]),
  f'witness {WIT}': avg([opchar(np.maximum(RK[(WIT[0], i)], RK[(WIT[1], j)])) for i in SEEDS for j in SEEDS]),
  f'engineering {ENG}': avg([opchar(np.maximum(RK[(ENG[0], k)], RK[(ENG[1], k)])) for k in SEEDS]),
  f'control {CTRL}': avg([opchar(np.maximum(RK[(CTRL[0], k)], RK[(CTRL[1], k)])) for k in SEEDS])}
print('6a operating characteristics at 25% (equal workload by construction):')
print(pd.DataFrame(OPCHAR).T.to_string())

# 6b. rho (validation) vs test advantage; concordant severe misses; model-level resampling
adv25 = {p: float(AC_T[p].mean(0)[T25]) for p in PAIRS}
def conc_sev(a, i, b, j):
    ma = np.zeros(n, bool); ma[np.argsort(-PT[(a, i)], kind='stable')[:R25]] = True
    mb = np.zeros(n, bool); mb[np.argsort(-PT[(b, j)], kind='stable')[:R25]] = True
    return int(((~ma) & (~mb) & (GRD_T >= 3)).sum())
conc = {p: float(np.mean([conc_sev(p[0], i, p[1], j) for i in SEEDS for j in SEEDS])) for p in PAIRS}
xs = np.array([RHO_VAL[p] for p in PAIRS])
r_adv = spearmanr(xs, [adv25[p] for p in PAIRS]).correlation
r_conc = spearmanr(xs, [conc[p] for p in PAIRS]).correlation
rng = np.random.default_rng(20260924); ml_adv, ml_conc = [], []
for _ in range(2000):
    samp = rng.choice(ARCHS, len(ARCHS), replace=True)
    ps = [(a, b) if (a, b) in adv25 else (b, a) for a, b in itertools.combinations(samp, 2) if a != b]
    if len(set(ps)) < 5: continue
    xx = [RHO_VAL[p] for p in ps]
    ml_adv.append(spearmanr(xx, [adv25[p] for p in ps]).correlation)
    ml_conc.append(spearmanr(xx, [conc[p] for p in ps]).correlation)
RHO_RES = {'spearman_rho_vs_test_advantage': round(float(r_adv), 3),
           'model_level_95ci_adv': [round(float(x), 3) for x in np.nanpercentile(ml_adv, [2.5, 97.5])],
           'spearman_rho_vs_concordant_severe_misses': round(float(r_conc), 3),
           'model_level_95ci_conc': [round(float(x), 3) for x in np.nanpercentile(ml_conc, [2.5, 97.5])]}
print('\n6b rho (validation) vs test outcomes:', RHO_RES)

# 6c. raw-probability OR-gate sensitivity
RAW = summarize(*run(ALL_T, PT, LAB_T, GRD_T, mode='raw')[:2])
print(f"\n6c raw-probability OR at 25%: {int(RAW['pairs_winning'][T25])}/55 win, mean adv {RAW['mean_adv'][T25]:.1f}")

# 6d. weight-invariance grid at 25%
grid = []
for w4, w3, w2 in LOCK['weight_invariance_grid']['miss_weight_sets']:
    for wfp in LOCK['weight_invariance_grid']['false_positive_weights']:
        s = summarize(*run(ALL_T, PT, LAB_T, GRD_T, w={'w4': w4, 'w3': w3, 'w2': w2, 'wfp': wfp})[:2])
        grid.append({'w4/w3/w2': f'{w4}/{w3}/{w2}', 'wfp': wfp, 'pairs_winning': int(s['pairs_winning'][T25]),
                     'mean_adv': round(float(s['mean_adv'][T25]), 1)})
GRID = pd.DataFrame(grid)
print('\n6d weight invariance at 25%:'); print(GRID.pivot(index='w4/w3/w2', columns='wfp', values='pairs_winning').to_string())

6a operating characteristics at 25% (equal workload by construction):
                                                            sensitivity  specificity     ppv     npv  referral_rate  severe_misses  grade4_misses
best_single (retfound_binary)                                    0.7241       0.8642  0.5623  0.9286           0.25         9.3333         0.0000
witness ('vit_base_5class', 'retfound_adversarial')              0.7351       0.8669  0.5709  0.9314           0.25         6.0000         0.0000
engineering ('densenet121_binary', 'retfound_adversarial')       0.7484       0.8701  0.5811  0.9349           0.25         5.6667         0.6667
control ('densenet121_binary', 'retfound_binary')                0.7472       0.8698  0.5802  0.9346           0.25         5.3333         0.3333

6b rho (validation) vs test outcomes: {'spearman_rho_vs_test_advantage': -0.322, 'model_level_95ci_adv': [-0.764, 0.16], 'spearman_rho_vs_concordant_severe_misses': 0.246, 'model_level_95ci_conc': [-

In [7]:
# Cell 7: headline tables and the curve figure
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

def curve_table(PTd, boot):
    rows = []
    for t, r in enumerate(BURDENS):
        row = {'burden': r}
        for k in ['pairs_winning','mean_adv','pairs_beating_roster','witness_adv_c','witness_adv_r','eng_adv','ctrl_adv','eng_minus_ctrl']:
            l, h = ci(boot[k]); row[k] = round(float(PTd[k][t]), 1); row[k + '_lo'] = round(float(l[t]), 1); row[k + '_hi'] = round(float(h[t]), 1)
        rows.append(row)
    return pd.DataFrame(rows)
CT = curve_table(PT_T, boot_T); CA = curve_table(PT_A, boot_A)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for C, lab, col in ((CT, 'EyePACS test', 'C0'), (CA, 'APTOS', 'C1')):
    x = C['burden'] * 100
    ax[0].plot(x, C['pairs_winning'], 'o-', color=col, label=lab)
    ax[0].fill_between(x, C['pairs_winning_lo'], C['pairs_winning_hi'], color=col, alpha=0.2)
    ax[1].plot(x, C['mean_adv'], 'o-', color=col, label=lab)
    ax[1].fill_between(x, C['mean_adv_lo'], C['mean_adv_hi'], color=col, alpha=0.2)
for a_ in ax: a_.axvline(PRIMARY * 100, ls=':', color='k'); a_.set_xlabel('Referral burden (%)'); a_.legend()
ax[0].set_ylabel('Pairs beating their better constituent (of 55)'); ax[0].axhline(27.5, ls='--', color='grey', lw=0.8)
ax[1].set_ylabel('Mean severity-weighted cost advantage'); ax[1].axhline(0, color='grey', lw=0.8)
fig.suptitle('Ensemble advantage at matched referral burden (rank OR-gate, 3 seeds, 95% bootstrap bands)')
fig.tight_layout(); fig.savefig('ensemble_advantage_curve_v2.png', dpi=200)
print('EyePACS test curve:'); print(CT[['burden','pairs_winning','pairs_winning_lo','pairs_winning_hi','mean_adv','mean_adv_lo','mean_adv_hi','pairs_beating_roster']].to_string(index=False))
print('\nAPTOS curve:'); print(CA[['burden','pairs_winning','pairs_winning_lo','pairs_winning_hi','mean_adv','mean_adv_lo','mean_adv_hi','pairs_beating_roster']].to_string(index=False))

EyePACS test curve:
 burden  pairs_winning  pairs_winning_lo  pairs_winning_hi  mean_adv  mean_adv_lo  mean_adv_hi  pairs_beating_roster
  0.100           32.0              14.0              38.0     106.8       -759.7        469.6                   5.0
  0.125           32.0              15.0              44.0     142.4       -731.7        615.2                  15.0
  0.150           44.0              26.0              48.0     582.1        -89.3        866.3                  32.0
  0.175           46.0              23.0              51.0     537.6       -112.5        859.0                  26.0
  0.200           43.0              23.0              50.0     435.7       -104.9        723.9                  21.0
  0.225           48.0              24.0              51.0     491.5        -53.3        723.0                  26.0
  0.250           52.0              23.0              52.0     472.0        -72.2        702.0                  34.0
  0.275           47.0              21.0    

In [8]:
# Cell 8: write results (one shot)
def at(C, k): r = C[C.burden == PRIMARY].iloc[0]; return [r[k], r[k + '_lo'], r[k + '_hi']]
RES = {
  'evaluated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'lock_sha256': LOCK_SHA, 'test_file_sha256': H_T, 'aptos_file_sha256': H_A,
  'primary_eyepacs_test_25pct': {
    'pairs_winning_of_55': at(CT, 'pairs_winning'), 'mean_advantage': at(CT, 'mean_adv'),
    'claim_supported': bool(at(CT, 'mean_adv')[1] > 0),
    'seed_combos_winning_of_495': int(PT_T['combos_winning'][T25]),
    'pairs_beating_best_roster_single_of_55': at(CT, 'pairs_beating_roster'),
    'best_roster_single': best_arch},
  'tier2_witness': {'pair': WIT, 'test_adv_vs_constituent': at(CT, 'witness_adv_c'),
                    'test_adv_vs_roster': at(CT, 'witness_adv_r'),
                    'validation_adv_vs_constituent': LOCK['tier2_witness']['validation_adv_vs_constituent'],
                    'winners_curse_shrinkage': round(1 - at(CT, 'witness_adv_c')[0] / LOCK['tier2_witness']['validation_adv_vs_constituent'], 3)},
  'tier3_engineering': {'pair': ENG, 'test_adv': at(CT, 'eng_adv'),
                        'control_pair': CTRL, 'control_test_adv': at(CT, 'ctrl_adv'),
                        'engineering_minus_control': at(CT, 'eng_minus_ctrl')},
  'operating_characteristics_25pct': OPCHAR, 'rho_analysis': RHO_RES,
  'raw_or_sensitivity_25pct': {'pairs_winning': int(RAW['pairs_winning'][T25]), 'mean_adv': round(float(RAW['mean_adv'][T25]), 1)},
  'weight_invariance_25pct': grid,
  'aptos_curve_note': 'APTOS is enriched (40.6% referable); tier 1 reported as a curve only',
}
with open('eval_results_v2.json', 'w') as f: json.dump(RES, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
CT.to_csv('eyepacs_test_curve_v2.csv', index=False); CA.to_csv('aptos_curve_v2.csv', index=False)
pd.DataFrame([{'model_a': p[0], 'model_b': p[1], 'rho_val': RHO_VAL[p], 'test_adv_25': round(adv25[p], 1),
               'concordant_severe_misses_25': round(conc[p], 2)} for p in PAIRS]).to_csv('eyepacs_test_pairs_25pct_v2.csv', index=False)
for fn, mime in (('eval_results_v2.json', 'application/json'), ('eyepacs_test_curve_v2.csv', 'text/csv'),
                 ('aptos_curve_v2.csv', 'text/csv'), ('eyepacs_test_pairs_25pct_v2.csv', 'text/csv')):
    up(fn, F_RESULTS, mime)
up('ensemble_advantage_curve_v2.png', F_FIGURES, 'image/png')
print(json.dumps({k: RES[k] for k in ['primary_eyepacs_test_25pct', 'tier2_witness', 'tier3_engineering']}, indent=2, default=str))

  uploaded eval_results_v2.json
  uploaded eyepacs_test_curve_v2.csv
  uploaded aptos_curve_v2.csv
  uploaded eyepacs_test_pairs_25pct_v2.csv
  uploaded ensemble_advantage_curve_v2.png
{
  "primary_eyepacs_test_25pct": {
    "pairs_winning_of_55": [
      52.0,
      23.0,
      52.0
    ],
    "mean_advantage": [
      472.0,
      -72.2,
      702.0
    ],
    "claim_supported": false,
    "seed_combos_winning_of_495": 417,
    "pairs_beating_best_roster_single_of_55": [
      34.0,
      3.0,
      41.0
    ],
    "best_roster_single": "retfound_binary"
  },
  "tier2_witness": {
    "pair": [
      "vit_base_5class",
      "retfound_adversarial"
    ],
    "test_adv_vs_constituent": [
      938.6,
      180.5,
      1634.2
    ],
    "test_adv_vs_roster": [
      716.2,
      -595.8,
      1086.2
    ],
    "validation_adv_vs_constituent": 1235.44,
    "winners_curse_shrinkage": 0.24
  },
  "tier3_engineering": {
    "pair": [
      "densenet121_binary",
      "retfound_adversarial"